# Merge Model: `qwen-1.5b`

Generated by [Lathe](https://github.com/lathe-agent/lathe). This notebook executes model merging with `mergekit` in the cloud with GPU/high-RAM acceleration, then uploads the merged safetensors directly to Hugging Face.


In [ ]:
# 1. Install mergekit and huggingface_hub
!pip install -q mergekit huggingface_hub


In [ ]:
# 2. Authenticate with Hugging Face
from huggingface_hub import login

login()


In [ ]:
# 3. Write mergekit configuration
config_yaml = """slices:
- sources:
  - model: Qwen/Qwen2.5-Coder-1.5B
    layer_range: [0, 28]
  - model: Qwen/Qwen2.5-1.5B-Instruct
    layer_range: [0, 28]
merge_method: slerp
base_model: Qwen/Qwen2.5-Coder-1.5B
parameters: {t: 0.5}
dtype: float16
"""

with open("config.yaml", "w", encoding="utf-8") as f:
    f.write(config_yaml)

print("Generated config.yaml successfully.")


In [ ]:
# 4. Run mergekit with lazy loading & low-memory flags
!mergekit-yaml config.yaml ./merged --lazy-unpickle --allow-crimes --low-cpu-memory --copy-tokenizer


In [ ]:
# 5. Upload merged model weights and tokenizer to Hugging Face
from huggingface_hub import HfApi

api = HfApi()
prompt_msg = "Enter destination HF repo id (e.g. username/qwen-1.5b): "\nrepo_id = input(prompt_msg).strip()
if repo_id:
    api.create_repo(repo_id=repo_id, exist_ok=True, private=True)
    api.upload_folder(
        folder_path="./merged",
        repo_id=repo_id,
        commit_message="Upload merged model qwen-1.5b generated via Lathe",
    )
    print(f"Uploaded successfully to https://huggingface.co/{repo_id}")
else:
    print("No destination repo ID provided; skipping push.")
